In [13]:
import kagglehub
import pandas as pd
import os
import ast
import html
import re
import unicodedata
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk 
nltk.download('punkt_tab')
nltk.download('stopwords')


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/jelenalazovic/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/jelenalazovic/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

Ucitavanje sirovih podataka

In [3]:
path = kagglehub.dataset_download("shuyangli94/foodcom-recipes-with-search-terms-and-tags")

In [4]:
csv_path = os.path.join(path, 'recipes_w_search_terms.csv')
df = pd.read_csv(csv_path)

In [5]:
def get_cuisine(x):
    if not isinstance(x, str):
        return None
    x_lower = x.lower()
    if 'italian' in x_lower:
        return 'Italian'
    if 'indian' in x_lower:
        return 'Indian'
    return None

italian_indian_df = df[df['search_terms'].apply(lambda x: isinstance(x, str) and ('italian' in x.lower() or 'indian' in x.lower()))].copy()
italian_indian_df['cuisine'] = italian_indian_df['search_terms'].apply(get_cuisine)
final_df = italian_indian_df[['name', 'steps', 'cuisine']].reset_index(drop=True)
final_df['steps'] = final_df['steps'].apply(lambda x: ' '.join(ast.literal_eval(x)) if isinstance(x, str) else x)
final_df['name'] = final_df['name'].str.lower()
final_df['steps'] = final_df['steps'].str.lower()

In [6]:
italian_sample = final_df[final_df['cuisine'] == 'Italian'].sample(n=10000, random_state=42)
indian_all = final_df[final_df['cuisine'] == 'Indian']

final_df = pd.concat([italian_sample, indian_all], ignore_index=True)
final_df.to_csv('./data/recipes_raw.csv')

Osnovne provere

In [ ]:
print('Number of null rows\n', final_df.isna().sum())
print('Number of duplicated rows', final_df.duplicated().sum())
df_no_dub = final_df.drop_duplicates().reset_index(drop=True)

Number of null rows
 name       0
steps      0
cuisine    0
dtype: int64
Number of duplicated rows 10


Čišćenje teksta (HTML entiteti, nevidljivi/kontrolni karakteri)

In [8]:
# zero-width space (200b), line/paragraph separator (2028/2029), BOM (feff), nbsp (a0)
_invisible = [0x200b, 0x2028, 0x2029, 0xfeff, 0xa0]
INVISIBLE_CHARS = re.compile('[' + ''.join(chr(c) for c in _invisible) + ']')
CONTROL_CHARS = re.compile('[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')

def clean_text(text):
    if not isinstance(text, str):
        return text
    text = html.unescape(text)  # &amp; -> &, &rsquo; -> ’, &eacute; -> é, ...
    text = unicodedata.normalize('NFKC', text)
    text = INVISIBLE_CHARS.sub(' ', text)
    text = CONTROL_CHARS.sub(' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_clean = df_no_dub.copy()
df_clean['name'] = df_clean['name'].apply(clean_text)
df_clean['steps'] = df_clean['steps'].apply(clean_text)

Tokenizacija

In [ ]:
df_clean['name_tokens'] = df_clean['name'].apply(word_tokenize)
df_clean['steps_tokens'] = df_clean['steps'].apply(word_tokenize)
df_clean.to_csv('./data/recipes_tokenized.csv')

Uklanjanje stop reči

In [ ]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [t for t in tokens if t not in stop_words]

df_clean['name_tokens'] = df_clean['name_tokens'].apply(remove_stopwords)
df_clean['steps_tokens'] = df_clean['steps_tokens'].apply(remove_stopwords)

Statisticka analiza podataka

In [23]:
df_clean['number_of_tokens'] = df_clean['steps_tokens'].apply(len)
df_clean['number_of_tokens'].describe()

count    16546.000000
mean        97.870482
std         62.917022
min          2.000000
25%         57.000000
50%         85.000000
75%        123.000000
max        949.000000
Name: number_of_tokens, dtype: float64

In [61]:
max_len = int(df_clean['number_of_tokens'].quantile(0.95))
df_clean_no_outliers = df_clean[
    df_clean['number_of_tokens'].between(13, max_len - 1)
]
print(df_clean_no_outliers['cuisine'].value_counts())
df_clean_no_outliers.to_csv('./data/recipes_cleaned.csv')

cuisine
Italian    9311
Indian     6201
Name: count, dtype: int64
